# 1. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# 2. Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

# 3. Silver Layer Transformations

## 3.1. Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## 3.2. Customer ID Cleanup

In [0]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

## 3.3. Country Normalization

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

## 3.4. Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
# Sanity Checks of Dataframe
df.limit(10).display()

# 4. Writing Table To Silver Layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

In [0]:
%sql
-- sanity Checks of Silver Table
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10